In [ ]:

"""
04_shap_maps.ipynb — 12-city SHAP spatial maps

For each city:
  - Load geometry from data/<city>_rq1_compare.gpkg
  - Join SHAP values + gap_z from modeling_table
  - Plot 4-panel map: gap_z + top-3 SHAP dims
  - Save → outputs/figures/city_shap_map/<city>_shap_map.png

Also moves existing shap_summary PNGs to outputs/figures/city_shap_summary/
"""

import geopandas as gpd, pandas as pd, numpy as np
import matplotlib, shutil, contextily as ctx
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

ROOT    = Path("..").resolve()   # rq1_explainability/
OUT     = ROOT / "outputs"
FIG_MAP = OUT / "figures" / "city_shap_map"
FIG_SUM = OUT / "figures" / "city_shap_summary"
FIG_MAP.mkdir(parents=True, exist_ok=True)
FIG_SUM.mkdir(parents=True, exist_ok=True)

# ── Load modeling table (for gap_z) ──────────────────────────────────────────
mt = pd.read_csv(OUT / "modeling_table.csv", dtype={"GEOID": str})
mt["GEOID"] = mt["GEOID"].str.zfill(11)

CITIES = sorted(mt["city"].unique().tolist())
EMBED_COLS = [f"A{i:02d}" for i in range(64)]

# ── SHAP map loop ─────────────────────────────────────────────────────────────
for city in CITIES:
    shap_path = OUT / f"{city}_shap_values.csv"
    gpkg_path = ROOT / "data" / f"{city}_rq1_compare.gpkg"

    if not shap_path.exists():
        print(f"{city}: no SHAP file, skip")
        continue
    if not gpkg_path.exists():
        print(f"{city}: no GPKG, skip")
        continue

    print(f"{city.upper()}")

    # Load geometry
    gdf = gpd.read_file(gpkg_path)[["GEOID", "geometry"]].copy()
    gdf["GEOID"] = gdf["GEOID"].astype(str).str.zfill(11)
    gdf = gdf.to_crs(4326)

    # Load SHAP values
    shap_df = pd.read_csv(shap_path, dtype={"GEOID": str})
    shap_df["GEOID"] = shap_df["GEOID"].str.zfill(11)

    # Top 3 SHAP dims
    shap_cols = [c for c in shap_df.columns if c != "GEOID"]
    top3 = shap_df[shap_cols].abs().mean().sort_values(ascending=False).head(3).index.tolist()
    print(f"  top SHAP dims: {top3}")

    # Merge: geometry + SHAP + gap_z
    city_mt = mt[mt["city"] == city][["GEOID", "hi_minus_lst_z"]].copy()
    joined = (gdf
              .merge(shap_df[["GEOID"] + top3], on="GEOID", how="left")
              .merge(city_mt, on="GEOID", how="left"))

    print(f"  joined rows: {len(joined)}, gap_z notna: {joined.hi_minus_lst_z.notna().sum()}")

    # ── 4-panel plot ──────────────────────────────────────────────────────────
    plot_cols   = ["hi_minus_lst_z"] + top3
    plot_titles = ["HI−LST gap (z-score)"] + [f"SHAP: {d}" for d in top3]

    fig, axes = plt.subplots(1, 4, figsize=(28, 7))
    fig.suptitle(f"{city.replace('_', ' ').title()} — SHAP Spatial Maps", fontsize=15, y=1.01)

    for ax, col, title in zip(axes, plot_cols, plot_titles):
        sub = joined.dropna(subset=[col])
        vmax = sub[col].abs().quantile(0.98)
        vmin = -vmax

        # background (all tracts grey)
        joined.plot(ax=ax, color="#e8e8e8", edgecolor="none")
        sub.plot(column=col, ax=ax, cmap="PuOr",
                 vmin=vmin, vmax=vmax, alpha=0.85,
                 legend=True, legend_kwds={"shrink": 0.6})
        try:
            ctx.add_basemap(ax, crs="EPSG:4326",
                            source=ctx.providers.OpenStreetMap.Mapnik,
                            alpha=0.4)
        except Exception as e:
            print(f"      basemap warning: {e}")
        ax.set_title(title, fontsize=11)
        ax.axis("off")

    plt.tight_layout()
    out_path = FIG_MAP / f"{city}_shap_map.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  saved → figures/city_shap_map/{city}_shap_map.png")

print(f"Maps   → {FIG_MAP}")
print(f"Summary plots → {FIG_SUM}")
